In [1]:
import pandas as pd
import lightgbm as lgb
import mlflow
import joblib, json, os
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, recall_score, precision_score

df = pd.read_csv("../data/customers_clean_v2.csv")
X = df.select_dtypes("number").drop(columns=["is_churn", "total_secs", "active_days", "unique_songs"], errors="ignore")
y = df["is_churn"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [2]:
mlflow.set_experiment("kkbox-churn")

settings = [
    {"num_leaves": 31, "learning_rate": 0.1, "n_estimators": 100},
    {"num_leaves": 63, "learning_rate": 0.05, "n_estimators": 300},
    {"num_leaves": 15, "learning_rate": 0.1, "n_estimators": 200},
]

best_auc, best_model = 0, None

for params in settings:
    with mlflow.start_run():
        model = lgb.LGBMClassifier(**params, random_state=42, verbose=-1)
        model.fit(X_train, y_train)
        prob = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, prob)

        mlflow.log_params(params)
        mlflow.log_metric("auc", auc)
        mlflow.log_metric("recall", recall_score(y_test, prob > 0.5))
        mlflow.log_metric("precision", precision_score(y_test, prob > 0.5))

        print(params, "AUC:", round(auc, 4))
        if auc > best_auc:
            best_auc, best_model = auc, model

2026/11/03 21:52:58 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/03 21:52:59 INFO mlflow.store.db.utils: Updating database tables
2026/11/03 21:53:02 INFO mlflow.tracking.fluent: Experiment with name 'kkbox-churn' does not exist. Creating a new experiment.


{'num_leaves': 31, 'learning_rate': 0.1, 'n_estimators': 100} AUC: 0.8712
{'num_leaves': 63, 'learning_rate': 0.05, 'n_estimators': 300} AUC: 0.8753
{'num_leaves': 15, 'learning_rate': 0.1, 'n_estimators': 200} AUC: 0.8699


In [3]:
os.makedirs("../models", exist_ok=True)
joblib.dump(best_model, "../models/churn_model.pkl")
json.dump(list(X.columns), open("../models/features.json", "w"))
print("Saved best model, AUC:", round(best_auc, 4))

Saved best model, AUC: 0.8753


In [4]:
print(X_test.iloc[0].to_json())

{"city":1.0,"age":null,"registered_via":7.0,"tenure_days":37.0,"plan_days":30.0,"auto_renew":1.0,"avg_discount":0.0,"ever_cancelled":0.0,"num_payments":1.0,"total_paid":100.0}
